<a href="https://colab.research.google.com/github/chiragbulbule/VaultLLM/blob/main/TenSEAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tenseal
import tenseal as ts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 43.9 MB/s eta 0:00:00


In [ ]:
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()
plain_number = [7.5]
encrypted = ts.ckks_vector(context, plain_number)
encrypted_result = encrypted * 2
result = encrypted_result.decrypt()
print(f"Original: 7.5 × 2 = 15.0")
print(f"FHE Result: {result[0]:.4f}")

Original: 7.5 × 2 = 15.0
FHE Result: 15.0000


In [ ]:
import tenseal as ts

# Same setup as before
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()

# A vector (pretend this is a tiny AI embedding)
plain_vector = [1.0, 2.0, 3.0, 4.0, 5.0]

# Encrypt the whole vector at once
encrypted_vector = ts.ckks_vector(context, plain_vector)

# Multiply every element by 2, while encrypted
encrypted_result = encrypted_vector * 2

# Decrypt
result = encrypted_result.decrypt()

print("Original:  ", plain_vector)
print("Expected:  ", [x * 2 for x in plain_vector])
print("FHE Result:", [round(x, 4) for x in result])

Original:   [1.0, 2.0, 3.0, 4.0, 5.0]
Expected:   [2.0, 4.0, 6.0, 8.0, 10.0]
FHE Result: [2.0, 4.0, 6.0, 8.0, 10.0]


In [ ]:
import tenseal as ts

context = ts.context(
    ts.SCHEME_TYPE.CKKS,
poly_modulus_degree=16384,
coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 40, 40, 60]  # 6 middle levels
)
context.global_scale = 2**40
context.generate_galois_keys()

plain_vector = [1.0, 2.0, 3.0, 4.0, 5.0]
encrypted_vector = ts.ckks_vector(context, plain_vector)

# Multiply but rescale between each operation
result = encrypted_vector * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2

decrypted = result.decrypt()

print("Original:       ", plain_vector)
print("Expected (×32): ", [x * 32 for x in plain_vector])
print("FHE Result:     ", [round(x, 4) for x in decrypted])

ValueError: scale out of bounds

In [ ]:
import tenseal as ts

# ============================================================
# SETUP (Cryptographer's job — your code)
# ============================================================
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=16384,
    coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 40, 40, 60]
)
context.global_scale = 2**40        # ← this line is missing in your cell
context.generate_galois_keys()
# ============================================================
# CLIENT SIDE (User's machine)
# ============================================================

# Step 1: User types a query
user_query = "The machine is overheating"
print(f"User types: '{user_query}'")

# Step 2: Convert to numbers (pretend embedding, AI engineer's job)
# In real project this would be a 768-dimension vector from a real model
# For now we simulate it with small numbers
plain_embedding = [0.23, -0.87, 0.45, 1.02, -0.33]
print(f"Converted to vector: {plain_embedding}")

# Step 3: Encrypt it (YOUR job)
encrypted_embedding = ts.ckks_vector(context, plain_embedding)
print(f"\nEncrypted (what server sees): {encrypted_embedding}")

# ============================================================
# SERVER SIDE (Blind — never sees real data)
# ============================================================
print("\n--- SERVER RECEIVES ENCRYPTED BLOB ---")
print("Server has NO idea what the original query was.")

# Step 4: Server does AI math on encrypted data
# In real project this would be encrypted matrix multiplication
# For now we simulate a simple weighted sum (like a tiny neural network layer)
weights = [0.5, 0.3, 0.8, 0.2, 0.6]  # pretend model weights
encrypted_result = encrypted_embedding.dot(weights)
print("Server computed result on ciphertext — still blind!")

# ============================================================
# BACK TO CLIENT SIDE
# ============================================================
print("\n--- RESULT RETURNS TO USER ---")

# Step 5: User decrypts
decrypted_result = encrypted_result.decrypt()
score = decrypted_result[0]
print(f"Decrypted score: {round(score, 4)}")

# Step 6: Interpret result (researcher/AI engineer's job)
if score > 0:
    print("Classification: ⚠️  ALERT — Negative signal detected")
else:
    print("Classification: ✅  OK — Normal signal")

User types: 'The machine is overheating'
Converted to vector: [0.23, -0.87, 0.45, 1.02, -0.33]

Encrypted (what server sees): <tenseal.tensors.ckksvector.CKKSVector object at 0x7fc218289850>

--- SERVER RECEIVES ENCRYPTED BLOB ---
Server has NO idea what the original query was.
Server computed result on ciphertext — still blind!

--- RESULT RETURNS TO USER ---
Decrypted score: 0.22
Classification: ⚠️  ALERT — Negative signal detected


In [ ]:
!pip install tenseal
import torch
import tenseal as ts
import pandas as pd
import random
from time import time

# those are optional and are not necessary for training
import numpy as np
import matplotlib.pyplot as plt

torch.random.manual_seed(73)
random.seed(73)


def split_train_test(x, y, test_ratio=0.3):
    idxs = [i for i in range(len(x))]
    random.shuffle(idxs)
    # delimiter between test and train data
    delim = int(len(x) * test_ratio)
    test_idxs, train_idxs = idxs[:delim], idxs[delim:]
    return x[train_idxs], y[train_idxs], x[test_idxs], y[test_idxs]

def random_data(m=1024, n=2):
    # data separable by the line `y = x`
    x_train = torch.randn(m, n)
    x_test = torch.randn(m // 2, n)
    y_train = (x_train[:, 0] >= x_train[:, 1]).float().unsqueeze(0).t()
    y_test = (x_test[:, 0] >= x_test[:, 1]).float().unsqueeze(0).t()
    return x_train, y_train, x_test, y_test

def heart_disease_data():
    data = pd.read_csv("./data/framingham.csv")
    # drop rows with missing values
    data = data.dropna()

    # drop some features
    data = data.drop(columns=["education", "id"])  # , "is_male" ??

    # binarize 'male' column
    data.male = data.male.apply(lambda x: 0 if x == "F" else 1)
    data = data.rename(columns={"male": "is_male"})

    # standarize data
    for col in data.columns:
        if col != "TenYearCHD":
            data[col] = data[col] - data[col].mean()
            data[col] = data[col] / data[col].std()

    # split into train/test
    data_x = torch.tensor(data.drop(columns=["TenYearCHD"]).values).float()
    data_y = torch.tensor(data["TenYearCHD"].values).float().unsqueeze(0).t()

    x_train, y_train, x_test, y_test = split_train_test(data_x, data_y)
    return x_train, y_train, x_test, y_test

# You can use whatever data you want without modification to the tutorial
x_train, y_train, x_test, y_test = random_data()
# x_train, y_train, x_test, y_test = heart_disease_data()

print("############# Data summary #############")
print(f"x_train has shape: {x_train.shape}")
print(f"y_train has shape: {y_train.shape}")
print(f"x_test has shape: {x_test.shape}")
print(f"y_test has shape: {y_test.shape}")
print("#######################################")

#logistic regression without encryption
class LR(torch.nn.Module):

    def __init__(self, n_features):
        super(LR, self).__init__()
        self.lr = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        out = torch.sigmoid(self.lr(x))
        return out

n_features = x_train.shape[1]
model = LR(n_features)
# use gradient descent with a learning_rate=1
optim = torch.optim.SGD(model.parameters(), lr=1)
# use Binary Cross Entropy Loss
criterion = torch.nn.BCELoss()
# define the number of epochs for both plain and encrypted training
EPOCHS = 5

def train(model, optim, criterion, x, y, epochs=EPOCHS):
    for e in range(1, epochs + 1):
        optim.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optim.step()
        print(f"Loss at epoch {e}: {loss.data}")
    return model

model = train(model, optim, criterion, x_train, y_train)


############# Data summary #############
x_train has shape: torch.Size([1024, 2])
y_train has shape: torch.Size([1024, 1])
x_test has shape: torch.Size([512, 2])
y_test has shape: torch.Size([512, 1])
#######################################
Loss at epoch 1: 0.904835045337677
Loss at epoch 2: 0.6667658090591431
Loss at epoch 3: 0.5325154066085815
Loss at epoch 4: 0.45469462871551514
Loss at epoch 5: 0.40461161732673645


In [ ]:
def accuracy(model, x, y):
    out = model(x)
    correct = torch.abs(y - out) < 0.5
    return correct.float().mean()

plain_accuracy = accuracy(model, x_test, y_test)
print(f"Accuracy on plain test_set: {plain_accuracy}")

Accuracy on plain test_set: 0.982421875


In [ ]:
#encrypted logistic regression
class EncryptedLR:

    def __init__(self, torch_lr):
        # TenSEAL processes lists and not torch tensors,
        # so we take out the parameters from the PyTorch model
        self.weight = torch_lr.lr.weight.data.tolist()[0]
        self.bias = torch_lr.lr.bias.data.tolist()

    def forward(self, enc_x):
        # We don't need to perform sigmoid as this model
        # will only be used for evaluation, and the label
        # can be deduced without applying sigmoid
        enc_out = enc_x.dot(self.weight) + self.bias
        return enc_out

    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    ################################################
    ## You can use the functions below to perform ##
    ## the evaluation with an encrypted model     ##
    ################################################

    def encrypt(self, context):
        self.weight = ts.ckks_vector(context, self.weight)
        self.bias = ts.ckks_vector(context, self.bias)

    def decrypt(self, context):
        self.weight = self.weight.decrypt()
        self.bias = self.bias.decrypt()


eelr = EncryptedLR(model)

In [ ]:
# parameters
poly_mod_degree = 4096
coeff_mod_bit_sizes = [40, 20, 40]
# create TenSEALContext
ctx_eval = ts.context(ts.SCHEME_TYPE.CKKS, poly_mod_degree, -1, coeff_mod_bit_sizes)
# scale of ciphertext to use
ctx_eval.global_scale = 2 ** 20
# this key is needed for doing dot-product operations
ctx_eval.generate_galois_keys()

In [ ]:
t_start = time()
enc_x_test = [ts.ckks_vector(ctx_eval, x.tolist()) for x in x_test]
t_end = time()
print(f"Encryption of the test-set took {int(t_end - t_start)} seconds")

Encryption of the test-set took 1 seconds


In [ ]:
# (optional) encrypt the model's parameters
eelr.encrypt(ctx_eval)

In [ ]:
def encrypted_evaluation(model, enc_x_test, y_test):
    t_start = time()

    correct = 0
    for enc_x, y in zip(enc_x_test, y_test):
        # encrypted evaluation
        enc_out = model(enc_x)
        # plain comparison
        out = enc_out.decrypt()
        out = torch.tensor(out)
        out = torch.sigmoid(out)
        if torch.abs(out - y) < 0.5:
            correct += 1

    t_end = time()
    print(f"Evaluated test_set of {len(x_test)} entries in {int(t_end - t_start)} seconds")
    print(f"Accuracy: {correct}/{len(x_test)} = {correct / len(x_test)}")
    return correct / len(x_test)


encrypted_accuracy = encrypted_evaluation(eelr, enc_x_test, y_test)
diff_accuracy = plain_accuracy - encrypted_accuracy
print(f"Difference between plain and encrypted accuracies: {diff_accuracy}")
if diff_accuracy < 0:
    print("Oh! We got a better accuracy on the encrypted test-set! The noise was on our side...")

Evaluated test_set of 512 entries in 1 seconds
Accuracy: 499/512 = 0.974609375
Difference between plain and encrypted accuracies: 0.0078125


In [ ]:
import tenseal as ts

def create_context():
    # Enough depth for a small classifier (3-4 multiplications)
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=16384,
        plain_modulus=-1, # Added this line
        coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    return context

def encrypt_vector(context, plain_vector):
    return ts.ckks_vector(context, plain_vector)

def decrypt_vector(encrypted_vector):
    return encrypted_vector.decrypt()


# ---- TEST IT ----
context = create_context()

# Simulate a fake embedding vector (384 numbers between -1 and 1)
import random
fake_embedding = [random.uniform(-1, 1) for _ in range(384)]

# Encrypt it
encrypted = encrypt_vector(context, fake_embedding)

# Decrypt it
decrypted = decrypt_vector(encrypted)

# Check accuracy
errors = [abs(fake_embedding[i] - decrypted[i]) for i in range(384)]
print(f"Max approximation error: {max(errors):.10f}")
print(f"Average error: {sum(errors)/len(errors):.10f}")
print(f"Encryption works on 384-dim vector: ✅")

Max approximation error: 0.0000000123
Average error: 0.0000000016
Encryption works on 384-dim vector: ✅


In [ ]:
# Given a list of 5 numbers, encrypt them, compute their average while encrypted
# decrypt and verify

import tenseal as ts

def create_context():
  context=ts.context(
      ts.SCHEME_TYPE.CKKS,
      poly_modulus_degree=8192,
      coeff_mod_bit_sizes=[60,40,60]
  )
  context.global_scale=2**40
vector=[10,20,30,40,50]
generate.galois_keys()
encrypted=ts.ckks_vector(context,vector)
print(encrypted)
decrypted=encrypted.decrypt()
print(decrypted)


NameError: name 'context' is not defined